# ============================================================
# GEOGRAPHIC INTERPOLATION OF ENDEMISM (GIE) EN R
# Raster de endemismo a 1 km desde un CSV con especies y coords
# ============================================================


In [19]:
# ----------------------------
# 1. Paquetes
# ----------------------------
libs <- c("terra", "sf", "dplyr", "readr", "purrr", "stringr", "tidyr")
invisible(lapply(libs, function(x) {
  if (!requireNamespace(x, quietly = TRUE)) install.packages(x)
  library(x, character.only = TRUE)
}))


In [20]:


# ----------------------------
# 2. Parámetros de usuario
# ----------------------------
ruta_csv <- "../../../DATOS/Datasets/Biodiversidad/biodiversidad_ocurrences_clean.csv"   # Cambiar por tu archivo
col_especie <- "scientificName"
col_lon     <- "decimalLongitude"
col_lat     <- "decimalLatitude"

# Resolución del raster final en metros
res_m <- 1000

# Número mínimo de registros por especie
min_registros <- 5

# Buffer externo adicional al AOI en metros
buffer_m <- 1000

# Esquema de clases inspirado en el paper
# Cortes superiores en km
breaks_km <- c(10, 20,50,100, 150)

# Directorio de salida
dir_salida <- "salidas_gie"
if (!dir.exists(dir_salida)) dir.create(dir_salida, recursive = TRUE)

In [21]:

# ----------------------------
# 3. Funciones auxiliares
# ----------------------------

# Elegir UTM automáticamente a partir del centro de los datos
utm_epsg_from_lonlat <- function(lon, lat) {
  zone <- floor((lon + 180) / 6) + 1
  epsg <- ifelse(lat >= 0, 32600, 32700) + zone
  epsg
}

# Estandarización 0-1 segura
minmax01 <- function(r) {
  mm <- terra::global(r, c("min", "max"), na.rm = TRUE)
  vmin <- mm[1, 1]
  vmax <- mm[1, 2]
  if (is.na(vmin) || is.na(vmax) || vmax <= vmin) {
    return(r * 0)
  }
  (r - vmin) / (vmax - vmin)
}

# Asignar clase según distancia máxima al centroide
clasificar_radio <- function(dist_km, breaks_km) {
  idx <- which(dist_km <= breaks_km)[1]
  if (is.na(idx)) idx <- length(breaks_km)
  tibble(
    clase_id = idx,
    clase_lbl = paste0("<= ", breaks_km[idx], " km"),
    radio_km = breaks_km[idx]
  )
}

# Índice de elongación simple a partir del bbox
# Sirve para alertar especies problemáticas
elongacion_bbox <- function(x, y) {
  rx <- diff(range(x, na.rm = TRUE))
  ry <- diff(range(y, na.rm = TRUE))
  mayor <- max(rx, ry)
  menor <- min(rx, ry)
  if (menor == 0) return(Inf)
  mayor / menor
}


replace_na_terra <- function(r, value = 0) {
  terra::ifel(is.na(r), value, r)
}

superficie_influencia <- function(template_r, xy, radio_m, aoi = NULL, sigma_factor = 3) {
  sigma <- radio_m / sigma_factor
  
  e <- terra::ext(
    xy[1] - radio_m, xy[1] + radio_m,
    xy[2] - radio_m, xy[2] + radio_m
  )
  
  # CORREGIDO: sin snap
  rloc <- terra::crop(template_r, e)
  
  p <- terra::vect(
    matrix(xy, ncol = 2),
    type = "points",
    crs = terra::crs(template_r)
  )
  
  d <- terra::distance(rloc, p)
  
  val <- exp(-0.5 * (d / sigma)^2)
  val[d > radio_m] <- NA
  
  if (!is.null(aoi)) {
    # CORREGIDO: sin snap
    aoi_local <- terra::crop(aoi, terra::ext(val))
    val <- terra::mask(val, aoi_local)
  }
  
  # Expandir al raster base
  val_full <- terra::extend(val, template_r)
  
  return(val_full)
}

In [22]:


# ----------------------------
# 4. Cargar y limpiar datos
# ----------------------------
df <- readr::read_csv(ruta_csv, show_col_types = FALSE) %>%
  dplyr::rename(
    species   = !!col_especie,
    longitude = !!col_lon,
    latitude  = !!col_lat
  ) %>%
  dplyr::mutate(
    species = stringr::str_trim(as.character(species)),
    longitude = as.numeric(longitude),
    latitude  = as.numeric(latitude)
  ) %>%
  dplyr::filter(
    !is.na(species),
    species != "",
    !is.na(longitude),
    !is.na(latitude),
    dplyr::between(longitude, -180, 180),
    dplyr::between(latitude, -90, 90)
  ) %>%
  dplyr::distinct(species, longitude, latitude, .keep_all = TRUE)%>%
    filter(coordinateUncertaintyInMeters <= 5000 | is.na(coordinateUncertaintyInMeters))


# Filtrar especies con mínimo de registros
sp_n <- df %>% dplyr::count(species, name = "n_reg")
df <- df %>%
  dplyr::left_join(sp_n, by = "species") %>%
  dplyr::filter(n_reg >= min_registros)

if (nrow(df) == 0) {
  stop("No quedan datos tras aplicar el filtro de registros mínimos.")
}


In [14]:

# ----------------------------
# 5. Convertir a objeto espacial
# ----------------------------
lon0 <- mean(df$longitude, na.rm = TRUE)
lat0 <- mean(df$latitude, na.rm = TRUE)
epsg_utm <- utm_epsg_from_lonlat(lon0, lat0)

pts_sf <- sf::st_as_sf(
  df,
  coords = c("longitude", "latitude"),
  crs = 4326,
  remove = FALSE
) %>%
  sf::st_transform(epsg_utm)

coords_m <- sf::st_coordinates(pts_sf)
df$x <- coords_m[, 1]
df$y <- coords_m[, 2]


In [15]:

# ----------------------------
# 6. Calcular centroides por especie y distancia máxima
# ----------------------------
sp_stats <- df %>%
  dplyr::group_by(species) %>%
  dplyr::summarise(
    n_reg = dplyr::first(n_reg),
    cx = mean(x, na.rm = TRUE),
    cy = mean(y, na.rm = TRUE),
    elongacion = elongacion_bbox(x, y),
    .groups = "drop"
  )

df_dist <- df %>%
  dplyr::left_join(sp_stats, by = c("species", "n_reg")) %>%
  dplyr::mutate(
    dist_centroide_m = sqrt((x - cx)^2 + (y - cy)^2)
  )

sp_stats <- df_dist %>%
  dplyr::group_by(species, n_reg, cx, cy, elongacion) %>%
  dplyr::summarise(
    max_dist_m = max(dist_centroide_m, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  dplyr::rowwise() %>%
  dplyr::bind_cols(
    clasificar_radio(.$max_dist_m / 1000, breaks_km)
  ) %>%
  dplyr::ungroup() %>%
  dplyr::mutate(
    radio_m = radio_km * 1000,
    flag_elongada = elongacion > 4
  )

# Guardar tabla de especies y métricas
readr::write_csv(sp_stats, file.path(dir_salida, "resumen_especies_gie.csv"))


Warning message:
In dist_km <= breaks_km :
  longer object length is not a multiple of shorter object length


In [16]:
# ----------------------------
# 7. Definir AOI y raster base
# ----------------------------
pts_vect <- terra::vect(pts_sf)

aoi <- terra::buffer(terra::convHull(pts_vect), width = buffer_m)

r_base <- terra::rast(
  ext = terra::ext(aoi),
  resolution = res_m,
  crs = terra::crs(pts_vect)
)

# Asignar valores base
terra::values(r_base) <- 0

# Enmascarar al AOI
r_base <- terra::mask(r_base, aoi)

In [17]:
# ----------------------------
# 8. Construir raster por categoría
# ----------------------------
categorias <- sort(unique(sp_stats$clase_id))

rasters_cat <- list()
rasters_cat_std <- list()

for (cid in categorias) {
  cat_info <- sp_stats %>% dplyr::filter(clase_id == cid)
  
  message("Procesando categoría ", cid, " con ", nrow(cat_info), " especies...")
  
  r_sum <- r_base
  terra::values(r_sum) <- 0
  names(r_sum) <- "base"
  
  for (i in seq_len(nrow(cat_info))) {
    xy <- c(cat_info$cx[i], cat_info$cy[i])
    radio_m <- cat_info$radio_m[i]
    
    r_sp <- superficie_influencia(
      template_r = r_base,
      xy = xy,
      radio_m = radio_m,
      aoi = aoi,
      sigma_factor = 3
    )
    
    r_sum <- replace_na_terra(r_sum, 0) + replace_na_terra(r_sp, 0)
  }
  
  r_sum[r_sum <= 0] <- NA
  r_sum <- terra::mask(r_sum, aoi)
  
  r_std <- minmax01(r_sum)
  r_std <- terra::mask(r_std, aoi)
  r_std[is.na(r_sum)] <- NA
  
  names(r_sum) <- paste0("gie_cat_", cid)
  names(r_std) <- paste0("gie_cat_", cid, "_std")
  
  rasters_cat[[as.character(cid)]] <- r_sum
  rasters_cat_std[[as.character(cid)]] <- r_std
  
  terra::writeRaster(
    r_sum,
    file.path(dir_salida, paste0("gie_categoria_", cid, "_raw.tif")),
    overwrite = TRUE
  )
  
  terra::writeRaster(
    r_std,
    file.path(dir_salida, paste0("gie_categoria_", cid, "_std.tif")),
    overwrite = TRUE
  )
}


Procesando categoría 4 con 4392 especies...


In [18]:

# ----------------------------
# 9. Raster consenso
# ----------------------------
# Suma de categorías estandarizadas, como en el paper
r_consenso <- rasters_cat_std[[1]]
if (length(rasters_cat_std) > 1) {
  for (j in 2:length(rasters_cat_std)) {
    r_consenso <- terra::cover(r_consenso, 0) + terra::cover(rasters_cat_std[[j]], 0)
  }
}
r_consenso[r_consenso <= 0] <- NA
names(r_consenso) <- "endemismo_consenso"

# Versión normalizada 0-1 del consenso para visualización
r_consenso_std <- minmax01(r_consenso)
r_consenso_std[is.na(r_consenso)] <- NA
names(r_consenso_std) <- "endemismo_consenso_std"

terra::writeRaster(
  r_consenso,
  file.path(dir_salida, "endemismo_consenso_1km.tif"),
  overwrite = TRUE
)

terra::writeRaster(
  r_consenso_std,
  file.path(dir_salida, "endemismo_consenso_1km_std.tif"),
  overwrite = TRUE
)

# ----------------------------
# 10. Isolíneas opcionales
# ----------------------------
# Útiles para delimitar áreas operacionales
niveles <- seq(0.1, 0.9, by = 0.1)
iso <- terra::as.contour(r_consenso_std, levels = niveles)
terra::writeVector(
  iso,
  file.path(dir_salida, "isolineas_endemismo.gpkg"),
  overwrite = TRUE
)

# ----------------------------
# 11. Gráficos rápidos
# ----------------------------
png(file.path(dir_salida, "mapa_endemismo_consenso.png"),
    width = 1800, height = 1400, res = 180)
plot(r_consenso_std,
     main = "Raster consenso de endemismo (GIE) - 1 km")
plot(aoi, add = TRUE, border = "grey30", lwd = 1)
points(sp_stats$cx, sp_stats$cy, pch = 16, cex = 0.3)
dev.off()

# ----------------------------
# 12. Resúmenes útiles
# ----------------------------
# Tabla de categorías
tabla_categorias <- sp_stats %>%
  dplyr::count(clase_id, clase_lbl, radio_km, name = "n_especies") %>%
  dplyr::arrange(clase_id)

readr::write_csv(tabla_categorias,
                 file.path(dir_salida, "categorias_gie.csv"))

# Especies potencialmente problemáticas
flags <- sp_stats %>%
  dplyr::filter(flag_elongada | n_reg < 5) %>%
  dplyr::arrange(desc(flag_elongada), n_reg)

readr::write_csv(flags,
                 file.path(dir_salida, "especies_revision_manual.csv"))

# ----------------------------
# 13. Mensaje final
# ----------------------------
cat("\nProceso completado.\n")
cat("EPSG UTM usado:", epsg_utm, "\n")
cat("Raster final:", file.path(dir_salida, "endemismo_consenso_1km.tif"), "\n")
cat("Raster final estandarizado:", file.path(dir_salida, "endemismo_consenso_1km_std.tif"), "\n")
cat("Isolíneas:", file.path(dir_salida, "isolineas_endemismo.gpkg"), "\n")


Proceso completado.
EPSG UTM usado: 32717 
Raster final: salidas_gie/endemismo_consenso_1km.tif 
Raster final estandarizado: salidas_gie/endemismo_consenso_1km_std.tif 
Isolíneas: salidas_gie/isolineas_endemismo.gpkg 
